# Anomaly check

QC on satellite GVF vs PhenoCam GCC and NDVI: scores table, gap by veg boxplots.

**Spin-up** (`gvf_sos == 1`): the phenology fit failed and landed on DOY 1 by
accident, not because green-up really started on Jan 1. On flat, low amplitude
curves (EN, sparse shrub, evergreen) there is no clear winter to summer swing, so
it pin SOS at the first day of data. That inflates gap /
divergence vs NDVI or GCC (noise misread as signal), so spin-up sites must be
flagged and usually excluded before interpreting lag or compression.

Artifacts: `anomaly_pipeline/output/` (`metadata/` scores, `boxplot/`,
`golden_standard_ranking.csv`).

**Veg Codes** DB = deciduous broadleaf, EN = evergreen needle, GR = grassland, AG = agriculture, SH = shrub


In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "anomaly_pipeline":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared.data_collection import (
    build_golden_ranking,
    collect_folder,
    group_summary,
    load_all_scores,
    load_table,
    plot_gap_boxplot_by_veg,
    top_n,
)

# GVF text in plotting stage (drop once, reuse here)
INPUT_DIR = REPO / "plotting_pipeline" / "input"
ANOMALY_DIR = REPO / "anomaly_pipeline" / "output"
METADATA_DIR = ANOMALY_DIR / "metadata"


## Score all folders

One row per site-year: SOS/MOS/DOS/EOS for GVF, GCC, and NDVI, plus pairwise
gap / DTW / divergence. Use it to find spin-up (`gvf_sos == 1`), large land-type
offsets, and other bad fits without opening every plot.

Scores every available input folder under `plotting_pipeline/input/` and writes
`anomaly_pipeline/output/metadata/<FOLDER>_scores.csv`. Missing folders are skipped.


In [2]:
FOLDERS = [
    # "GBOV_2023",
    # "GBOV_2024",
    # "GoldenSites_2023",
    # "GoldenSites_2024",
    "Satellite_GVF_timeseries_2023",
    "Satellite_GVF_timeseries_2024",
]
LIMIT = None
SORT = "gvf_vs_ndvi_div"
TOP = 10

score_paths = {}
for folder in FOLDERS:
    src = INPUT_DIR / folder
    if not src.is_dir():
        print(f"skip {folder}: no folder at {src}")
        continue
    csv_path = collect_folder(folder, INPUT_DIR, ANOMALY_DIR, limit=LIMIT)
    score_paths[folder] = csv_path
    df = load_table(csv_path)
    spin = int(df["gvf_sos"].eq(1.0).sum()) if "gvf_sos" in df.columns else 0
    print(f"{folder}: {len(df)} rows | spin-up={spin} | {csv_path.name}")


  BART (DB, 2023): GVF-GCC div=1.75  GVF-NDVI div=1.44  GCC-NDVI div=1.55
  HARV (DB, 2023): GVF-GCC div=5.30  GVF-NDVI div=1.23  GCC-NDVI div=6.26
  HARV (UN, 2023): GVF-GCC div=2.06  GVF-NDVI div=1.59  GCC-NDVI div=1.33
  BLAN (DB, 2023): GVF-GCC div=3.02  GVF-NDVI div=2.53  GCC-NDVI div=0.67
  BLAN (EN, 2023): GVF-GCC div=2.68  GVF-NDVI div=3.02  GCC-NDVI div=2.52
  SCBI (DB, 2023): GVF-GCC div=1.89  GVF-NDVI div=1.42  GCC-NDVI div=0.57
  JERC (DB, 2023): GVF-GCC div=2.53  GVF-NDVI div=2.36  GCC-NDVI div=0.94
  JERC (EN, 2023): GVF-GCC div=2.15  GVF-NDVI div=2.08  GCC-NDVI div=1.96
  JERC (UN, 2023): GVF-GCC div=2.78  GVF-NDVI div=2.39  GCC-NDVI div=0.78
  KING (WL, 2023): GVF-GCC div=1.22  GVF-NDVI div=0.88  GCC-NDVI div=1.19
  UKFS (DB, 2023): GVF-GCC div=1.65  GVF-NDVI div=2.02  GCC-NDVI div=0.79
  UKFS (UN, 2023): GVF-GCC div=1.22  GVF-NDVI div=1.64  GCC-NDVI div=0.84
  MLBS (UN, 2023): GVF-GCC div=2.10  GVF-NDVI div=1.66  GCC-NDVI div=1.23
  TALL (EN, 2023): GVF-GCC div=2.68  G

In [3]:
# peek: top rows + veg summary for each scored folder
for folder, csv_path in score_paths.items():
    df = load_table(csv_path)
    cols = [
        "site", "lag", "greenup_comp", "senescence_comp", "veg", "year",
        "gvf_sos", "gcc_sos", "ndvi_sos",
        "gvf_vs_ndvi_div", "gvf_vs_ndvi_gap", "gvf_vs_ndvi_dtw",
        "gvf_vs_gcc_div", "gcc_vs_ndvi_div",
    ]
    cols = [c for c in cols if c in df.columns]
    print(f"\n=== {folder} ===")
    display(top_n(df, by=SORT, n=TOP)[cols])
    display(group_summary(df, by="veg"))



=== Satellite_GVF_timeseries_2023 ===


,site,lag,greenup_comp,senescence_comp,veg,year,gvf_sos,gcc_sos,ndvi_sos,gvf_vs_ndvi_div,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw,gvf_vs_gcc_div,gcc_vs_ndvi_div
0,bbc1,-8.0,5.2500,0.6548,DB,2023,101.0,122.0,109.0,0.518588,6.50,0.054302,1.984858,1.538137
1,blackrockforest,-1.0,1.5417,1.0577,DB,2023,113.0,113.0,114.0,0.553339,7.25,0.035481,0.553215,0.697705
2,arsbrooks10,-2.0,1.4516,0.5500,AG,2023,137.0,141.0,139.0,0.567715,7.50,0.032001,1.227026,0.709935
3,arscolessouth,-4.0,1.5000,3.2941,AG,2023,140.0,154.0,144.0,0.578394,7.50,0.042680,1.020370,0.760096
4,arsbrooks11,-13.0,2.1429,1.1000,AG,2023,137.0,151.0,150.0,0.640380,8.50,0.033237,0.877663,0.525808
5,robinson2,-1.0,2.8235,0.8311,DB,2023,95.0,98.0,96.0,0.670589,9.00,0.027732,1.168259,1.035840
6,willowcreek,-4.0,3.3077,2.3600,DB,2023,116.0,128.0,120.0,0.721060,9.75,0.024631,1.357718,1.170697
7,morganmonroe2,14.0,2.1000,0.6304,DB,2023,98.0,93.0,84.0,0.781215,10.50,0.031215,1.505663,1.911052
8,arscolesnorth,-3.0,1.4118,1.1429,AG,2023,140.0,145.0,143.0,0.794892,10.75,0.027035,0.864995,0.643768
9,meadpasturese,21.0,1.0222,0.7400,AG,2023,62.0,54.0,41.0,0.798364,10.25,0.066221,0.870810,0.843729


,veg,gvf_vs_gcc_div,gvf_vs_gcc_gap,gvf_vs_gcc_dtw,gvf_vs_ndvi_div,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw,gcc_vs_ndvi_div,gcc_vs_ndvi_gap,gcc_vs_ndvi_dtw
0,AG,2.140344,29.306818,0.047000,1.831598,24.977273,0.047507,1.046350,14.056818,0.042291
1,DB,1.897148,25.925000,0.045362,1.906311,26.033333,0.046787,1.996830,27.308333,0.046235
2,EB,2.294672,31.000000,0.080387,1.741225,23.250000,0.080511,1.796305,24.750000,0.028448
3,EN,2.761791,37.630952,0.073865,3.836412,52.130952,0.112773,3.018076,40.809524,0.103110
4,GR,3.392784,46.035714,0.104519,3.655147,50.128571,0.074535,2.707883,36.778571,0.080842
5,SH,3.961381,54.370690,0.077760,3.900823,53.448276,0.083089,2.494129,33.853448,0.076026
6,UN,2.046222,27.861111,0.056143,1.922363,25.944444,0.069189,1.484011,19.805556,0.069329
7,WL,2.212738,30.200000,0.055595,1.682542,22.750000,0.057542,1.983919,27.050000,0.051776
8,XX,2.515408,34.208333,0.071956,2.524310,34.083333,0.089787,2.531332,34.291667,0.081927



=== Satellite_GVF_timeseries_2024 ===


,site,lag,greenup_comp,senescence_comp,veg,year,gvf_sos,gcc_sos,ndvi_sos,gvf_vs_ndvi_div,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw,gvf_vs_gcc_div,gcc_vs_ndvi_div
0,GRSM,-1.0,1.7500,1.2549,DB,2024,94.0,93.0,95.0,0.275970,3.50,0.025970,0.612219,0.783873
1,SJER,3.0,1.0588,1.8750,GR,2024,22.0,16.0,19.0,0.440175,5.75,0.029461,0.736259,0.303684
2,arsbrooks10,-4.0,1.2571,4.0833,AG,2024,142.0,147.0,146.0,0.514735,6.75,0.032593,0.832705,0.734621
3,KONZ,-2.0,2.0000,1.5588,GR,2024,98.0,102.0,100.0,0.618303,8.25,0.029017,1.289456,0.675801
4,homesteadsprings,-8.0,1.4688,1.4216,DB,2024,90.0,98.0,98.0,0.634923,8.25,0.045637,1.057548,0.634294
5,GRSM,-2.0,2.4706,0.5470,UN,2024,94.0,97.0,96.0,0.673078,8.75,0.048078,1.704239,1.140907
6,sedgwick2,-3.0,0.8947,0.9067,GR,2024,12.0,9.0,15.0,0.682497,9.25,0.021783,0.831283,0.451758
7,KONZ,-4.0,2.1379,1.3947,GR,2024,98.0,105.0,102.0,0.721947,9.50,0.043376,1.166843,0.528005
8,SRER,15.0,0.6667,1.0513,SH,2024,39.0,19.0,24.0,0.743644,9.75,0.047216,0.673141,0.299410
9,snodgrass4,-24.0,4.4615,1.8462,UN,2024,126.0,158.0,150.0,0.747048,10.00,0.032763,1.635113,1.054882


,veg,gvf_vs_gcc_div,gvf_vs_gcc_gap,gvf_vs_gcc_dtw,gvf_vs_ndvi_div,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw,gcc_vs_ndvi_div,gcc_vs_ndvi_gap,gcc_vs_ndvi_dtw
0,AG,2.199535,30.066667,0.051916,2.326339,31.833333,0.052530,1.722085,23.500000,0.043514
1,DB,2.210798,30.270000,0.048656,2.237413,30.190000,0.080984,1.478648,19.780000,0.065791
2,DN,4.741099,65.250000,0.080385,3.546953,48.500000,0.082667,1.765415,24.250000,0.033272
3,EB,3.421400,46.625000,0.091043,4.199810,57.625000,0.083739,3.233664,44.000000,0.090807
4,EN,3.551331,48.535714,0.084494,3.346266,45.130952,0.122627,3.180803,43.047619,0.105973
5,GR,3.757457,51.222222,0.098727,5.017886,68.701389,0.110644,3.412855,46.493056,0.091923
6,SH,2.955545,40.294643,0.077357,4.393242,60.160714,0.096048,3.536584,48.312500,0.085691
7,TN,4.455419,60.916667,0.104228,3.355575,45.916667,0.075813,1.751095,24.000000,0.036810
8,UN,2.473901,33.846154,0.056319,2.049098,27.865385,0.058714,1.763588,23.942308,0.053423
9,WL,2.993861,40.843750,0.076450,2.593749,35.500000,0.058035,2.257265,30.843750,0.054140


## Satellite Data Gap boxplot by veg

Distribution of `gvf_vs_ndvi_gap` by vegetation type for **every** scores CSV.
Spin-up sites are red diamonds so we can see how much they inflate the apparent
discrepancy (especially EN / GR; DB barely moves).

True lag / compression examples and the DB-vs-shrub/mixed effect-size test are
in the section after golden ranking (same clean, non-spin-up pool).

Writes `anomaly_pipeline/output/boxplot/<FOLDER>_BOXPLOT.png`.


In [4]:
boxplot_paths = []
for csv_path in sorted(METADATA_DIR.glob("*_scores.csv")):
    out = plot_gap_boxplot_by_veg(csv_path, ANOMALY_DIR)
    boxplot_paths.append(out)
    print(out)


Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2023_BOXPLOT.png
/Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2023_BOXPLOT.png
Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2024_BOXPLOT.png
/Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2024_BOXPLOT.png
Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GoldenSites_2023_BOXPLOT.png
/Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GoldenSites_2023_BOXPLOT.png
Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GoldenSites

## Golden standard ranking

Drop spin-up, then rank sites by combined GVF-GCC / GVF-NDVI divergence (gap +
DTW). Closed-canopy **DB** sites are flagged as the control group: most uniform
at VIIRS scales, tightest cross-product agreement, almost no spin-up. Their
gap/DTW distribution is the irreducible baseline under ideal conditions.

Top ranks ≈ small disagreement (baseline); mid/lower ranks still include
larger offsets. The next section pulls lag/compression examples from metadata
and tests whether shrub/mixed gap exceeds the DB baseline (Cohen's d).

Needs scores under `output/metadata/`. Writes `output/golden_standard_ranking.csv`.


In [5]:
rank_path = build_golden_ranking(ANOMALY_DIR)
rank = load_table(rank_path)
rank_cols = [
    "rank", "site", "veg", "year", "source", "golden_candidate",
    "combined_div", "combined_gap", "combined_dtw",
]
rank_cols = [c for c in rank_cols if c in rank.columns]
print(rank_path, "|", len(rank), "rows |", int(rank["golden_candidate"].sum()), "DB candidates")
#overall combined score is the sum of gap, div, and dtw
display(rank.head(15)[rank_cols]) #top 15 overall best combined score in any veg
# display(rank.loc[rank["golden_candidate"]].head(15)[rank_cols]) #top 15 DB candidates


Ranked 334 site-years (79 DB golden candidates); excluded spin-up=True
Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/golden_standard_ranking.csv
/Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/golden_standard_ranking.csv | 334 rows | 79 DB candidates


,rank,site,veg,year,source,golden_candidate,combined_div,combined_gap,combined_dtw
0,1,GRSM,DB,2024,Satellite_GVF_timeseries_2024,True,0.444095,5.875,0.024452
1,2,blackrockforest,DB,2023,GoldenSites_2023,True,0.553277,7.250,0.035420
2,3,blackrockforest,DB,2023,Satellite_GVF_timeseries_2023,True,0.553277,7.250,0.035420
3,4,SJER,GR,2024,Satellite_GVF_timeseries_2024,False,0.588217,7.750,0.034646
4,5,arsbrooks10,AG,2024,Satellite_GVF_timeseries_2024,False,0.673720,9.125,0.021935
5,6,SRER,SH,2024,GBOV_2024,False,0.708393,9.125,0.056607
6,7,SRER,SH,2024,Satellite_GVF_timeseries_2024,False,0.708393,9.125,0.056607
7,8,sedgwick2,GR,2024,Satellite_GVF_timeseries_2024,False,0.756890,10.250,0.024747
8,9,arsbrooks11,AG,2023,Satellite_GVF_timeseries_2023,False,0.759021,10.250,0.026878
9,10,arscolessouth,AG,2023,Satellite_GVF_timeseries_2023,False,0.799382,10.625,0.040453


## Lag, compression, and effect size vs DB baseline

Same clean pool as ranking (spin-up excluded). Two related questions in one pass:

1. **Examples from the CSVs** spotting lag/compression in
   `metadata/` (and why they are not at the top of
   `golden_standard_ranking.csv`):
   - **Lag-ish** (per phase): `lag_sos` / `lag_mos` / `lag_dos` / `lag_eos`
     = `gvf_* − ndvi_*` (``lag`` still means SOS). Look for large |lag| with a
     plausible green-up span (not spin-up).
      - Positive: GVF after NDVI (GVF later)
      - 0: same DOY
      - Negative: GVF before NDVI (GVF earlier)
      - |lag| guide: ~0–15 typical | 15–40 worth a look | 40+ lag candidate.
   - **Compression-ish (`greenup_comp`):** green-up length ratio
     `(gvf_mos−gvf_sos)/(gcc_mos−gcc_sos)` 
      - ratio = 1 -> GVF's green-up phase took exactly as many days as GCC's. No stretching, no squeezing
      - ratio < 1 -> the numerator (GVF's duration) is smaller than the denominator (GCC's duration). GVF's green-up happened in fewer days than GCC's and GVF is compressed relative to GCC.
      - ratio > 1 ->  GVF's duration is bigger. GVF took longer to go from onset to peak than GCC did, GVF is stretched relative to GCC.
   - **Senescence** (`senescence_comp`): same idea for DOS→EOS, `(gvf_eos−gvf_dos)/(gcc_eos−gcc_dos)` (=1 same length, <1 GVF shorter/compressed, >1 GVF longer/stretched).

2. **Effect-size test** — is shrub/mixed (`SH`+`GR`+`EN`) `gvf_vs_ndvi_gap`
   larger than the DB golden-standard mean? Cohen's d + one-sided Welch t-test
   (`H1: mixed > DB`). Only *excess* beyond the DB baseline supports a
   land-cover-driven lag claim.

Caveat: small `n` for shrub/mixed means the test can be underpowered; treat a
non-significant result as "not yet demonstrated," not proof of no effect.


### CODE: Shared helpers

Load all scores, drop spin-up, and define lollipop helpers + Cohen's d used by
the GoldenSites / GBOV cells below.


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import Image, display
from scipy import stats

# --- shared clean pool (used by GoldenSites + GBOV cells) ---
all_scores = load_all_scores(ANOMALY_DIR)
all_scores["spin_up"] = all_scores["gvf_sos"].eq(1.0)
clean = all_scores.loc[~all_scores["spin_up"]].copy()
print(
    f"rows={len(all_scores)} | spin-up={int(all_scores['spin_up'].sum())} | "
    f"clean={len(clean)} | sources={sorted(all_scores['source'].unique())}"
)

show_cols = [
    c for c in [
        "site", "lag", "lag_sos", "lag_mos", "lag_dos", "lag_eos",
        "greenup_comp", "senescence_comp", "veg", "year", "source",
        "gvf_sos", "ndvi_sos", "gvf_mos", "ndvi_mos",
        "gvf_dos", "ndvi_dos", "gvf_eos", "ndvi_eos",
        "gcc_sos", "gcc_mos", "gcc_dos", "gcc_eos",
        "gvf_vs_ndvi_gap", "gvf_vs_ndvi_dtw",
    ] if c in clean.columns
]


def _lollipop(ax, labels, values, color, ref_line=None):
    """Horizontal lollipop chart (cleaner than thick bars for ranked site values)."""
    y = np.arange(len(labels))
    vals = np.asarray(values, dtype=float)
    ax.hlines(y, 0 if ref_line is None else ref_line, vals, color=color, alpha=0.55, linewidth=1.4)
    ax.scatter(vals, y, color=color, s=42, zorder=3, edgecolors="white", linewidths=0.4)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=7)
    if ref_line is not None:
        ax.axvline(ref_line, color="black", linestyle="--", linewidth=1, alpha=0.75)
    ax.grid(True, axis="x", alpha=0.3)


def plot_lag_ranked_by_veg(df: pd.DataFrame, out_png: Path, title_prefix: str, show: bool = True):
    """Lag by phase (SOS/MOS/DOS/EOS) × veg as lollipops.

    Rows = phases, columns = veg types. Phase label sits at the top-right of each row.
    lag_* = gvf_* − ndvi_* (positive => GVF later).
    """
    phases = [
        ("lag_sos", "SOS"),
        ("lag_mos", "MOS"),
        ("lag_dos", "DOS"),
        ("lag_eos", "EOS"),
    ]
    plot_df = df.copy()
    # tolerate older frames that only have ``lag``
    if "lag_sos" not in plot_df.columns and "lag" in plot_df.columns:
        plot_df["lag_sos"] = plot_df["lag"]

    vegs = sorted(plot_df["veg"].dropna().unique())
    n_veg = max(len(vegs), 1)
    max_n = 4
    for col, _ in phases:
        if col in plot_df.columns and plot_df[col].notna().any():
            max_n = max(max_n, int(plot_df.dropna(subset=[col]).groupby("veg").size().max()))

    fig, axes = plt.subplots(
        len(phases),
        n_veg,
        figsize=(5.8 * n_veg, max(3.4, 0.32 * max_n + 1.6) * len(phases)),
        sharex=False,
        squeeze=False,
    )
    colors = plt.cm.tab10(np.linspace(0, 1, max(n_veg, 1)))

    for row_i, (col, phase) in enumerate(phases):
        for col_i, (veg, color) in enumerate(zip(vegs, colors)):
            ax = axes[row_i][col_i]
            if col not in plot_df.columns:
                ax.set_visible(False)
                continue
            sub = (
                plot_df.loc[plot_df["veg"].eq(veg)]
                .dropna(subset=[col, "site"])
                .sort_values(col, ascending=True)
            )
            if sub.empty:
                ax.set_visible(False)
                continue
            _lollipop(ax, list(sub["site"]), sub[col].values, color, ref_line=0)
            ax.tick_params(axis="y", pad=6)
            if row_i == 0:
                ax.set_title(f"{veg} (n={len(sub)})")
            ax.set_xlabel(f"{phase} lag (days)")

        # phase label in the right margin (top of each row), clear of the panels
        right_ax = axes[row_i][-1]
        right_ax.text(
            1.28,
            0.95,
            phase,
            transform=right_ax.transAxes,
            ha="left",
            va="top",
            fontsize=12,
            fontweight="bold",
            clip_on=False,
        )

    fig.suptitle(f"{title_prefix} — Phase lag (GVF − NDVI) by veg", y=0.995)
    # large bottom/right margins: legend below EOS, phase tags clear of last column
    fig.subplots_adjust(left=0.08, right=0.90, top=0.93, bottom=0.22, hspace=0.75, wspace=1.15)
    fig.legend(
        handles=[
            Line2D([0], [0], color="none", label="lag_sos/mos/dos/eos = gvf_* − ndvi_*"),
            Line2D([0], [0], color="none", label="+ : GVF after NDVI (GVF later)"),
            Line2D([0], [0], color="none", label="0 : same DOY"),
            Line2D([0], [0], color="none", label="− : GVF before NDVI (GVF earlier)"),
            Line2D([0], [0], color="none", label="|lag| guide: ~0–15 typical | 15–40 look | 40+ candidate"),
        ],
        loc="upper center",
        bbox_to_anchor=(0.47, 0.18),
        ncol=1,
        frameon=True,
        fontsize=8,
    )
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight", pad_inches=0.5)
    plt.close(fig)
    if show:
        display(Image(filename=str(out_png)))
    print(f"Wrote {out_png}")


def plot_compression_ranked_by_veg(df: pd.DataFrame, out_png: Path, title_prefix: str, show: bool = True):
    """greenup_comp | senescence_comp as separate side-by-side lollipop panels per veg."""
    gu_color = "#4C78A8"
    sen_color = "#F58518"
    metrics = [
        ("greenup_comp", gu_color, "greenup_comp = (gvf_mos−gvf_sos)/(gcc_mos−gcc_sos)"),
        ("senescence_comp", sen_color, "senescence_comp = (gvf_eos−gvf_dos)/(gcc_eos−gcc_dos)"),
    ]
    vegs = sorted(df["veg"].dropna().unique())
    n_veg = max(len(vegs), 1)
    max_n = 4
    for col, _, _ in metrics:
        if col in df.columns and df[col].notna().any():
            max_n = max(max_n, int(df.dropna(subset=[col]).groupby("veg").size().max()))

    fig, axes = plt.subplots(
        n_veg, 2,
        figsize=(13, max(3.8, 0.40 * max_n + 1.8) * n_veg),
        sharex=False,
        squeeze=False,
    )
    for row, veg in enumerate(vegs):
        for col_i, (col, color, _) in enumerate(metrics):
            ax = axes[row][col_i]
            sub = df.loc[df["veg"].eq(veg)].dropna(subset=[col, "site"]).sort_values(col, ascending=True)
            if sub.empty:
                ax.set_visible(False)
                continue
            _lollipop(ax, list(sub["site"]), sub[col].values, color, ref_line=1)
            ax.set_xlabel(col)
            if col_i == 0:
                ax.set_ylabel(veg)
            ax.set_title(f"{veg} · {col} (n={len(sub)})")

    fig.suptitle(
        f"{title_prefix} — greenup_comp (left) | senescence_comp (right)",
        y=1.01,
    )
    fig.legend(
        handles=[
            Line2D([0], [0], color=gu_color, lw=6, label=metrics[0][2]),
            Line2D([0], [0], color=sen_color, lw=6, label=metrics[1][2]),
            Line2D([0], [0], color="none", label="ratio = 1 : same length as GCC"),
            Line2D([0], [0], color="none", label="ratio < 1 : GVF shorter (compressed vs GCC)"),
            Line2D([0], [0], color="none", label="ratio > 1 : GVF longer (stretched vs GCC)"),
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=1,
        frameon=True,
        fontsize=8,
    )
    fig.tight_layout(rect=[0.0, 0.08, 1.0, 0.97])
    fig.subplots_adjust(hspace=0.85, wspace=0.45)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight", pad_inches=0.4)
    plt.close(fig)
    if show:
        display(Image(filename=str(out_png)))
    print(f"Wrote {out_png}")


def cohens_d(a: pd.Series, b: pd.Series) -> float:
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return float("nan")
    var_p = ((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2)
    return (b.mean() - a.mean()) / np.sqrt(var_p)


rows=386 | spin-up=52 | clean=334 | sources=['GBOV_2023', 'GBOV_2024', 'GoldenSites_2023', 'GoldenSites_2024', 'Satellite_GVF_timeseries_2023', 'Satellite_GVF_timeseries_2024']


### GoldenSites 2023

Tables with `lag` / `greenup_comp` / `senescence_comp` for **GoldenSites_2023**
(spin-up excluded). Lollipop PNGs are saved under `output/lollipopPlot/` (not shown here).


In [7]:
gs = clean.loc[clean["source"].eq("GoldenSites_2023")].copy()
print(f"GoldenSites_2023 clean n={len(gs)}:")
if gs.empty:
    print("No clean rows; skip.")
else:
    display(gs[show_cols])

    lag_png = ANOMALY_DIR / "lollipopPlot" / "2023_GoldenSite_Lag.png"
    comp_png = ANOMALY_DIR / "lollipopPlot" / "2023_GoldenSite_Compression.png"
    plot_lag_ranked_by_veg(gs, lag_png, "2023 GoldenSites", show=False)
    plot_compression_ranked_by_veg(gs, comp_png, "2023 GoldenSites", show=False)

    # effect size on full clean pool (DB vs shrub/mixed across sources)
    db_gap = clean.loc[clean["veg"].eq("DB"), "gvf_vs_ndvi_gap"].dropna()
    mixed_gap = clean.loc[clean["veg"].isin(["SH", "GR", "EN"]), "gvf_vs_ndvi_gap"].dropna()

    d = cohens_d(db_gap, mixed_gap)
    tt = stats.ttest_ind(mixed_gap, db_gap, equal_var=False, alternative="greater")

    print("\nEffect size: shrub/mixed (SH+GR+EN) vs DB golden baseline (gvf_vs_ndvi_gap)")
    print(f"  DB mean gap:      {db_gap.mean():.1f} days (n={len(db_gap)})")
    print(f"  Shrub/mixed mean: {mixed_gap.mean():.1f} days (n={len(mixed_gap)})")
    print(f"  Cohen's d:        {d:.2f}  (|d|<0.2 negligible, ~0.5 medium, ~0.8 large)")
    print(f"  one-sided p:      {tt.pvalue:.3f}  (H1: mixed > DB)")
    if tt.pvalue >= 0.05:
        print(
            "  -> not significant: after dropping spin-up, shrub/mixed does not show a "
            "detectable excess lag over DB (underpowered if n_mixed is small)."
        )
    else:
        print(
            "  -> significant excess gap in shrub/mixed beyond the DB baseline "
            "(still check n and spin-up screening before claiming ecology)."
        )


GoldenSites_2023 clean n=22:


,site,lag,lag_sos,lag_mos,lag_dos,lag_eos,greenup_comp,senescence_comp,veg,year,...,gvf_dos,ndvi_dos,gvf_eos,ndvi_eos,gcc_sos,gcc_mos,gcc_dos,gcc_eos,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw
23,blackrockforest,-1.0,-1.0,-1.0,-17.0,10.0,1.5417,1.0577,DB,2023,...,221.0,238.0,331.0,321.0,113.0,137.0,216.0,320.0,7.25,0.035481
24,robinson2,-1.0,-1.0,17.0,-8.0,10.0,2.8235,0.8311,DB,2023,...,226.0,234.0,349.0,339.0,98.0,115.0,197.0,345.0,9.00,0.027732
25,willowcreek,-4.0,-4.0,9.0,18.0,8.0,3.3077,2.3600,DB,2023,...,247.0,229.0,306.0,298.0,128.0,141.0,244.0,269.0,9.75,0.024631
26,morganmonroe2,14.0,14.0,-4.0,5.0,19.0,2.1000,0.6304,DB,2023,...,253.0,248.0,340.0,321.0,93.0,113.0,202.0,340.0,10.50,0.031215
27,dukehw,-5.0,-5.0,22.0,4.0,21.0,2.4138,0.8803,DB,2023,...,243.0,239.0,346.0,325.0,72.0,101.0,214.0,331.0,13.00,0.029432
28,bigtraillake,11.0,11.0,41.0,-4.0,-2.0,1.3051,1.1039,EN,2023,...,218.0,222.0,303.0,305.0,105.0,164.0,224.0,301.0,14.50,0.062200
29,cafcookeastltar01,-4.0,-4.0,32.0,15.0,14.0,5.0000,3.5625,AG,2023,...,186.0,171.0,243.0,229.0,119.0,133.0,181.0,197.0,16.25,0.023524
30,arkansaswhitaker,17.0,17.0,11.0,22.0,27.0,1.4194,1.1667,AG,2023,...,227.0,205.0,283.0,256.0,131.0,162.0,203.0,251.0,19.25,0.037298
31,SCBI,-11.0,-11.0,30.0,-11.0,26.0,3.8333,1.9310,DB,2023,...,237.0,248.0,349.0,323.0,87.0,105.0,261.0,319.0,19.50,0.027887
32,coweeta,-26.0,-26.0,18.0,-21.0,24.0,3.6400,0.6089,DB,2023,...,236.0,257.0,345.0,321.0,97.0,122.0,163.0,342.0,22.25,0.041029


Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2023_GoldenSite_Lag.png
Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2023_GoldenSite_Compression.png

Effect size: shrub/mixed (SH+GR+EN) vs DB golden baseline (gvf_vs_ndvi_gap)
  DB mean gap:      25.7 days (n=79)
  Shrub/mixed mean: 54.0 days (n=160)
  Cohen's d:        1.07  (|d|<0.2 negligible, ~0.5 medium, ~0.8 large)
  one-sided p:      0.000  (H1: mixed > DB)
  -> significant excess gap in shrub/mixed beyond the DB baseline (still check n and spin-up screening before claiming ecology).


### GoldenSites 2024

Tables with `lag` / `greenup_comp` / `senescence_comp` for **GoldenSites_2024**
(spin-up excluded). Lollipop PNGs are saved under `output/lollipopPlot/` (not shown here).

Lollipop plots omit veg codes **EN**, **WL**, and **UN**.


In [8]:
gs24 = clean.loc[clean["source"].eq("GoldenSites_2024")].copy()
print(f"GoldenSites_2024 clean n={len(gs24)}:")
if gs24.empty:
    print("No clean (non-spin-up) rows for GoldenSites_2024; skip plots/tests.")
else:
    display(gs24[show_cols])

    # omit WL / UN from lollipop visualizations only
    gs24_plot = gs24.loc[~gs24["veg"].isin(["EN", "WL", "UN"])].copy()
    print(f"  plot n={len(gs24_plot)} after dropping EN/WL/UN")

    lag_png = ANOMALY_DIR / "lollipopPlot" / "2024_GoldenSite_Lag.png"
    comp_png = ANOMALY_DIR / "lollipopPlot" / "2024_GoldenSite_Compression.png"
    plot_lag_ranked_by_veg(gs24_plot, lag_png, "2024 GoldenSites", show=False)
    plot_compression_ranked_by_veg(gs24_plot, comp_png, "2024 GoldenSites", show=False)

    # effect size within GoldenSites_2024 only
    db_gap = gs24.loc[gs24["veg"].eq("DB"), "gvf_vs_ndvi_gap"].dropna()
    mixed_gap = gs24.loc[gs24["veg"].isin(["SH", "GR", "EN"]), "gvf_vs_ndvi_gap"].dropna()

    d = cohens_d(db_gap, mixed_gap)
    tt = stats.ttest_ind(mixed_gap, db_gap, equal_var=False, alternative="greater")

    print("\nEffect size (GoldenSites_2024 only): shrub/mixed (SH+GR+EN) vs DB (gvf_vs_ndvi_gap)")
    print(f"  DB mean gap:      {db_gap.mean():.1f} days (n={len(db_gap)})")
    print(f"  Shrub/mixed mean: {mixed_gap.mean():.1f} days (n={len(mixed_gap)})")
    print(f"  Cohen's d:        {d:.2f}  (|d|<0.2 negligible, ~0.5 medium, ~0.8 large)")
    print(f"  one-sided p:      {tt.pvalue:.3f}  (H1: mixed > DB)")
    if len(db_gap) < 2 or len(mixed_gap) < 2:
        print("  -> too few sites in one group for a stable test; treat descriptively.")
    elif tt.pvalue >= 0.05:
        print(
            "  -> not significant: after dropping spin-up, shrub/mixed does not show a "
            "detectable excess lag over DB (underpowered if n_mixed is small)."
        )
    else:
        print(
            "  -> significant excess gap in shrub/mixed beyond the DB baseline "
            "(still check n and spin-up screening before claiming ecology)."
        )


GoldenSites_2024 clean n=16:


,site,lag,lag_sos,lag_mos,lag_dos,lag_eos,greenup_comp,senescence_comp,veg,year,...,gvf_dos,ndvi_dos,gvf_eos,ndvi_eos,gcc_sos,gcc_mos,gcc_dos,gcc_eos,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw
50,willowcreek,3.0,3.0,16.0,12.0,-9.0,3.4667,2.7778,DB,2024,...,249.0,237.0,324.0,333.0,126.0,141.0,259.0,286.0,10.00,0.036982
51,lostcreek,44.0,44.0,0.0,10.0,11.0,2.2308,1.2529,WL,2024,...,231.0,221.0,340.0,329.0,123.0,149.0,213.0,300.0,16.25,0.038452
52,arsope3ltar,12.0,12.0,-22.0,20.0,14.0,0.9216,0.8936,AG,2024,...,270.0,250.0,312.0,298.0,2.0,206.0,246.0,293.0,17.00,0.039292
54,STEI,52.0,52.0,-9.0,-27.0,-4.0,5.0769,2.8810,UN,2024,...,215.0,242.0,336.0,340.0,130.0,143.0,246.0,288.0,23.00,0.036153
55,arsmorris2,69.0,69.0,7.0,-1.0,-12.0,0.7004,2.0526,AG,2024,...,243.0,244.0,321.0,333.0,2.0,239.0,254.0,292.0,22.25,0.095034
56,shalehillsczo,-20.0,-20.0,-6.0,-35.0,36.0,2.0968,2.1549,DB,2024,...,192.0,227.0,345.0,309.0,106.0,137.0,233.0,304.0,24.25,0.031418
57,blackrockforest,-14.0,-14.0,-2.0,-17.0,61.0,4.2000,1.4792,DB,2024,...,202.0,219.0,344.0,283.0,120.0,130.0,188.0,284.0,23.50,0.147099
58,coweeta,-14.0,-14.0,7.0,-53.0,31.0,3.2500,0.9803,DB,2024,...,200.0,253.0,349.0,318.0,104.0,120.0,183.0,335.0,26.25,0.034305
59,russellsage,-16.0,-16.0,31.0,59.0,1.0,2.6154,1.6870,DB,2024,...,172.0,113.0,366.0,365.0,77.0,103.0,250.0,365.0,26.75,0.212313
60,uiefmiscanthus2,-33.0,-33.0,47.0,-41.0,9.0,5.0690,3.0870,AG,2024,...,242.0,283.0,313.0,304.0,122.0,151.0,279.0,302.0,32.50,0.033393


  plot n=13 after dropping EN/WL/UN
Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2024_GoldenSite_Lag.png
Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2024_GoldenSite_Compression.png

Effect size (GoldenSites_2024 only): shrub/mixed (SH+GR+EN) vs DB (gvf_vs_ndvi_gap)
  DB mean gap:      22.1 days (n=5)
  Shrub/mixed mean: 57.7 days (n=6)
  Cohen's d:        2.00  (|d|<0.2 negligible, ~0.5 medium, ~0.8 large)
  one-sided p:      0.006  (H1: mixed > DB)
  -> significant excess gap in shrub/mixed beyond the DB baseline (still check n and spin-up screening before claiming ecology).


### GBOV 2023 and GBOV 2024

Tables with `lag` / `greenup_comp` / `senescence_comp` for **GBOV_2023** and
**GBOV_2024** (spin-up excluded). Lollipop PNGs are saved under `output/lollipopPlot/`
(not shown here).


In [9]:
for source, file_tag, title in [
    ("GBOV_2023", "2023_GBOV", "2023 GBOV"),
    ("GBOV_2024", "2024_GBOV", "2024 GBOV"),
]:
    subset = clean.loc[clean["source"].eq(source)].copy()
    print(f"\n=== {source} clean n={len(subset)} ===")
    if subset.empty:
        print(f"No clean rows for {source}; skip.")
        continue

    display(subset[show_cols])

    lag_png = ANOMALY_DIR / "lollipopPlot" / f"{file_tag}_Lag.png"
    comp_png = ANOMALY_DIR / "lollipopPlot" / f"{file_tag}_Compression.png"
    plot_lag_ranked_by_veg(subset, lag_png, title, show=False)
    plot_compression_ranked_by_veg(subset, comp_png, title, show=False)



=== GBOV_2023 clean n=10 ===


,site,lag,lag_sos,lag_mos,lag_dos,lag_eos,greenup_comp,senescence_comp,veg,year,...,gvf_dos,ndvi_dos,gvf_eos,ndvi_eos,gcc_sos,gcc_mos,gcc_dos,gcc_eos,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw
0,HARV,-5.0,-5.0,14.0,-45.0,13.0,NaN,0.7902,DB,2023,...,221.0,266.0,334.0,321.0,2.0,2.0,212.0,355.0,19.25,0.060059
1,BART,-3.0,-3.0,16.0,-34.0,26.0,8.8750,1.3452,DB,2023,...,221.0,255.0,334.0,308.0,125.0,133.0,219.0,303.0,19.75,0.032221
3,DELA,-9.0,-9.0,34.0,-44.0,-7.0,2.7188,0.6123,DB,2023,...,211.0,255.0,350.0,357.0,63.0,95.0,138.0,365.0,23.50,0.023523
4,CPER,16.0,16.0,34.0,36.0,31.0,1.9143,1.1553,GR,2023,...,194.0,158.0,313.0,282.0,115.0,150.0,161.0,264.0,29.25,0.037971
5,KONA,-65.0,-65.0,-16.0,-18.0,22.0,3.8696,1.6957,AG,2023,...,215.0,233.0,293.0,271.0,163.0,186.0,226.0,272.0,30.25,0.028389
6,ORNL,-17.0,-17.0,39.0,-39.0,35.0,3.8182,3.2273,DB,2023,...,207.0,246.0,349.0,314.0,84.0,106.0,258.0,302.0,32.50,0.266771
7,ONAQ,83.0,83.0,-22.0,41.0,0.0,1.1765,2.2429,SH,2023,...,208.0,167.0,365.0,365.0,79.0,130.0,230.0,300.0,36.50,0.094593
9,TALL,74.0,74.0,-103.0,-36.0,-12.0,0.3730,1.3298,EN,2023,...,212.0,248.0,337.0,349.0,31.0,216.0,243.0,337.0,56.25,0.080673
10,JORN,58.0,58.0,105.0,141.0,-26.0,0.3630,1.5591,GR,2023,...,194.0,53.0,339.0,365.0,2.0,272.0,272.0,365.0,82.50,0.126758
11,STER,79.0,79.0,137.0,141.0,133.0,NaN,NaN,AG,2023,...,176.0,35.0,231.0,98.0,NaN,NaN,NaN,NaN,122.50,0.137996


Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2023_GBOV_Lag.png
Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2023_GBOV_Compression.png

=== GBOV_2024 clean n=11 ===


,site,lag,lag_sos,lag_mos,lag_dos,lag_eos,greenup_comp,senescence_comp,veg,year,...,gvf_dos,ndvi_dos,gvf_eos,ndvi_eos,gcc_sos,gcc_mos,gcc_dos,gcc_eos,gvf_vs_ndvi_gap,gvf_vs_ndvi_dtw
12,SRER,15.0,15.0,-10.0,-13.0,1.0,0.6667,1.0513,SH,2024,...,79.0,92.0,366.0,365.0,19.0,79.0,92.0,365.0,9.75,0.047216
13,BART,3.0,3.0,13.0,-17.0,10.0,5.7778,1.5902,DB,2024,...,229.0,246.0,326.0,316.0,130.0,139.0,235.0,296.0,10.75,0.044301
14,HARV,-5.0,-5.0,17.0,-14.0,-8.0,4.3333,0.7886,DB,2024,...,229.0,243.0,326.0,334.0,128.0,140.0,211.0,334.0,11.00,0.035721
15,ORNL,-43.0,-43.0,37.0,11.0,2.0,5.5238,1.0440,DB,2024,...,235.0,224.0,330.0,328.0,87.0,108.0,231.0,322.0,23.25,0.045148
16,DELA,-4.0,-4.0,19.0,-77.0,1.0,3.4375,0.8954,DB,2024,...,152.0,229.0,366.0,365.0,79.0,95.0,126.0,365.0,25.25,0.033498
17,MOAB,-33.0,-33.0,10.0,60.0,-23.0,0.6203,2.5385,GR,2024,...,239.0,179.0,305.0,328.0,2.0,239.0,247.0,273.0,31.50,0.069665
18,ONAQ,24.0,24.0,-4.0,-13.0,-98.0,1.3521,0.6497,SH,2024,...,139.0,152.0,267.0,365.0,42.0,113.0,113.0,310.0,34.75,0.078202
19,KONA,15.0,15.0,52.0,102.0,42.0,1.9242,0.6242,AG,2024,...,221.0,119.0,314.0,272.0,42.0,108.0,128.0,277.0,52.75,0.046400
20,TALL,75.0,75.0,-58.0,-83.0,1.0,0.2995,2.6548,EN,2024,...,143.0,226.0,366.0,365.0,11.0,218.0,278.0,362.0,54.25,0.062004
21,JORN,76.0,76.0,46.0,127.0,-32.0,0.3714,0.6275,GR,2024,...,237.0,110.0,333.0,365.0,2.0,212.0,212.0,365.0,70.25,0.130059


Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2024_GBOV_Lag.png
Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/2024_GBOV_Compression.png


### Combined lag: GBOV + GoldenSites (2023 vs 2024)

One figure for **AG / DB / GR / SH** only. Rows = SOS / MOS / DOS / EOS.
If the same site appears in both years, draw two lollipops close together
(**2023 = blue**, **2024 = green**); different sites are spaced farther apart.
X-axis is always centered at 0 (symmetric limits from the panel max |lag|).


In [10]:
# Combined lag across GBOV + GoldenSites; pair 2023/2024 for the same site
VEG_KEEP = ["AG", "DB", "GR", "SH"]
SOURCES = [
    "GBOV_2023", "GBOV_2024",
    "GoldenSites_2023", "GoldenSites_2024",
]
YEAR_COLOR = {2023: "#1f77b4", 2024: "#2ca02c"}  # blue / green
PHASES = [
    ("lag_sos", "SOS"),
    ("lag_mos", "MOS"),
    ("lag_dos", "DOS"),
    ("lag_eos", "EOS"),
]

# within-site pair gap (small) vs between-site gap (larger)
PAIR_HALF = 0.20
SITE_STEP = 1.55

def _sym_axis_lim(vals, pad=5.0, empty=5.0) -> float:
    """Half-range: max(|neg|, |pos|) + pad; keeps 0 centered."""
    if not vals:
        return float(empty)
    return float(max(abs(v) for v in vals) + pad)


def _apply_centered_xlim(ax, lim: float) -> None:
    """Symmetric xlim around 0 with readable tick steps (25/50/...)."""
    from matplotlib.ticker import MultipleLocator

    ax.set_xlim(-lim, lim)
    ax.set_autoscalex_on(False)
    # choose step so each side has a few ticks (e.g. 0,25,50 or 0,50,100,...)
    step = 5.0
    for candidate in (5, 10, 25, 50, 100, 200, 500):
        if lim / candidate <= 5:
            step = float(candidate)
            break
    ax.xaxis.set_major_locator(MultipleLocator(step))



def plot_combined_lag_years(
    df: pd.DataFrame,
    out_png: Path,
    title_prefix: str = "GBOV + GoldenSites",
    show: bool = False,
):
    """Phase × veg lag lollipops; 2023/2024 paired for the same site."""
    plot_df = df.copy()
    if "lag_sos" not in plot_df.columns and "lag" in plot_df.columns:
        plot_df["lag_sos"] = plot_df["lag"]
    if "year" not in plot_df.columns:
        plot_df["year"] = plot_df["source"].str.extract(r"(20\d{2})")[0].astype(float)

    vegs = [v for v in VEG_KEEP if v in set(plot_df["veg"].dropna())]
    n_veg = max(len(vegs), 1)

    # precompute site order + y layout per veg (shared across phase rows)
    layout = {}
    max_sites = 1
    for veg in vegs:
        sub = plot_df.loc[plot_df["veg"].eq(veg)]
        sites = sorted(sub["site"].dropna().unique())
        max_sites = max(max_sites, len(sites))
        y_centers = []
        y_tick_pos = []
        y_tick_lab = []
        y = 0.0
        for site in sites:
            yrs = sorted({int(y) for y in sub.loc[sub["site"].eq(site), "year"].dropna()})
            y_centers.append((site, yrs, y))
            y_tick_pos.append(y)
            y_tick_lab.append(site)
            y += SITE_STEP
        layout[veg] = {
            "centers": y_centers,
            "ticks": y_tick_pos,
            "labels": y_tick_lab,
            "ymax": max(y - SITE_STEP, 0.0),
        }

    fig, axes = plt.subplots(
        len(PHASES),
        n_veg,
        figsize=(5.8 * n_veg, max(4.0, 0.48 * max_sites + 2.0) * len(PHASES)),
        sharex=False,
        squeeze=False,
    )

    for row_i, (col, phase) in enumerate(PHASES):
        for col_i, veg in enumerate(vegs):
            ax = axes[row_i][col_i]
            info = layout[veg]
            sub = plot_df.loc[plot_df["veg"].eq(veg)].copy()
            sub["_year_i"] = pd.to_numeric(sub["year"], errors="coerce").astype("Int64")

            drawn = []
            for site, yrs, y0 in info["centers"]:
                site_rows = sub.loc[sub["site"].eq(site)]
                if 2023 in yrs and 2024 in yrs:
                    year_y = {2023: y0 - PAIR_HALF, 2024: y0 + PAIR_HALF}
                elif yrs:
                    year_y = {yrs[0]: y0}
                else:
                    year_y = {}

                for year, yy in year_y.items():
                    r = site_rows.loc[site_rows["_year_i"].eq(year)]
                    if r.empty or col not in r.columns:
                        continue
                    val = r.iloc[0][col]
                    if pd.isna(val):
                        continue
                    drawn.append((yy, float(val), year))

            for yy, val, year in drawn:
                color = YEAR_COLOR.get(year, "#555555")
                ax.hlines(yy, 0, val, color=color, alpha=0.65, linewidth=1.6)
                ax.scatter(
                    [val], [yy], color=color, s=40, zorder=3,
                    edgecolors="white", linewidths=0.4,
                )

            ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.8)
            ax.set_yticks(info["ticks"])
            ax.set_yticklabels(info["labels"], fontsize=7)
            ax.set_ylim(-0.55, info["ymax"] + 0.55)
            ax.invert_yaxis()
            ax.grid(True, axis="x", alpha=0.3)

            # per-panel: max(|neg|,|pos|) + 5 days — AG can be ~50; DB/SH can be >100
            lim = _sym_axis_lim([v for _, v, _ in drawn], pad=5.0, empty=5.0)
            _apply_centered_xlim(ax, lim)

            if row_i == 0:
                ax.set_title(f"{veg} (n_sites={len(info['labels'])})")
            ax.set_xlabel(f"{phase} lag (days)")

        right_ax = axes[row_i][-1]
        right_ax.text(
            1.20, 0.95, phase,
            transform=right_ax.transAxes,
            ha="left", va="top", fontsize=12, fontweight="bold", clip_on=False,
        )

    fig.suptitle(
        f"{title_prefix} — Phase lag 2023 (blue) vs 2024 (green)",
        y=0.995,
    )
    fig.subplots_adjust(left=0.09, right=0.90, top=0.93, bottom=0.20, hspace=0.80, wspace=1.05)
    fig.legend(
        handles=[
            Line2D([0], [0], color=YEAR_COLOR[2023], lw=3, marker="o", label="2023"),
            Line2D([0], [0], color=YEAR_COLOR[2024], lw=3, marker="o", label="2024"),
            Line2D([0], [0], color="none", label="same site: paired lines close together"),
            Line2D([0], [0], color="none", label="lag_* = gvf_* − ndvi_*  |  0 centered on each panel"),
            Line2D([0], [0], color="none", label="+ later GVF | − earlier GVF"),
        ],
        loc="upper center",
        bbox_to_anchor=(0.47, 0.16),
        ncol=1,
        frameon=True,
        fontsize=8,
    )
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight", pad_inches=0.45)
    plt.close(fig)
    if show:
        display(Image(filename=str(out_png)))
    print(f"Wrote {out_png}")


combined = clean.loc[
    clean["source"].isin(SOURCES) & clean["veg"].isin(VEG_KEEP)
].copy()
# label GBOV sites with a GBOV_ prefix (GoldenSites keep bare names)
combined["site_label"] = combined["site"].astype(str)
gbov_mask = combined["source"].str.startswith("GBOV_")
combined.loc[gbov_mask, "site_label"] = "GBOV_" + combined.loc[gbov_mask, "site_label"]
print(
    f"combined pool n={len(combined)} | sites={combined['site_label'].nunique()} | "
    f"veg={sorted(combined['veg'].unique())}"
)

out = ANOMALY_DIR / "lollipopPlot" / "Combined" / "Lag_2023_2024.png"
# plot uses site_label when present
_plot_df = combined.copy()
if "site_label" in _plot_df.columns:
    _plot_df["site"] = _plot_df["site_label"]
plot_combined_lag_years(_plot_df, out, show=False)


combined pool n=53 | sites=39 | veg=['AG', 'DB', 'GR', 'SH']
Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/Combined/Lag_2023_2024.png


### Combined compression: GBOV + GoldenSites (2023 vs 2024)

Same pool/rules as combined lag (**AG / DB / GR / SH**; `GBOV_` prefix;
**2023 = blue**, **2024 = green**; same-site pairs close together).

Axis is centered at **0** (0 = same length as GCC, + stretched, − compressed).
Rows = greenup_comp / senescence_comp.


In [11]:
# Combined compression (ratio-1), 2023 blue / 2024 green, 0-centered
# (self-contained: rebuilds the same AG/DB/GR/SH pool as the combined-lag cell)
VEG_KEEP = ["AG", "DB", "GR", "SH"]
SOURCES = [
    "GBOV_2023", "GBOV_2024",
    "GoldenSites_2023", "GoldenSites_2024",
]
YEAR_COLOR = {2023: "#1f77b4", 2024: "#2ca02c"}
PAIR_HALF = 0.20
SITE_STEP = 1.55

def _sym_axis_lim(vals, pad=5.0, empty=5.0) -> float:
    """Half-range: max(|neg|, |pos|) + pad; keeps 0 centered."""
    if not vals:
        return float(empty)
    return float(max(abs(v) for v in vals) + pad)


def _apply_centered_xlim(ax, lim: float) -> None:
    """Symmetric xlim around 0 with readable tick steps for compression deltas."""
    from matplotlib.ticker import MultipleLocator

    ax.set_xlim(-lim, lim)
    ax.set_autoscalex_on(False)
    # e.g. 0,0.5,1 or 0,1,2 or 0,5,10 — mirrors lag's "small vs large" behavior
    step = 0.5
    for candidate in (0.5, 1, 2, 5, 10, 25, 50):
        if lim / candidate <= 5:
            step = float(candidate)
            break
    ax.xaxis.set_major_locator(MultipleLocator(step))


combined = clean.loc[
    clean["source"].isin(SOURCES) & clean["veg"].isin(VEG_KEEP)
].copy()
combined["site_label"] = combined["site"].astype(str)
gbov_mask = combined["source"].str.startswith("GBOV_")
combined.loc[gbov_mask, "site_label"] = "GBOV_" + combined.loc[gbov_mask, "site_label"]
_plot_df = combined.copy()
_plot_df["site"] = _plot_df["site_label"]
print(
    f"combined compression pool n={len(_plot_df)} | sites={_plot_df['site'].nunique()} | "
    f"veg={sorted(_plot_df['veg'].unique())}"
)

COMP_METRICS = [
    ("greenup_comp", "greenup_comp", "greenup = (gvf_mos−gvf_sos)/(gcc_mos−gcc_sos)"),
    ("senescence_comp", "senescence_comp", "senescence = (gvf_eos−gvf_dos)/(gcc_eos−gcc_dos)"),
]


def plot_combined_compression_years(
    df: pd.DataFrame,
    out_png: Path,
    title_prefix: str = "GBOV + GoldenSites",
    show: bool = False,
):
    """Compression × veg lollipops; plot (ratio-1) so 0 is centered."""
    plot_df = df.copy()
    if "year" not in plot_df.columns:
        plot_df["year"] = plot_df["source"].str.extract(r"(20\d{2})")[0].astype(float)

    vegs = [v for v in VEG_KEEP if v in set(plot_df["veg"].dropna())]
    n_veg = max(len(vegs), 1)

    layout = {}
    max_sites = 1
    for veg in vegs:
        sub = plot_df.loc[plot_df["veg"].eq(veg)]
        sites = sorted(sub["site"].dropna().unique())
        max_sites = max(max_sites, len(sites))
        y_centers = []
        y_tick_pos = []
        y_tick_lab = []
        y = 0.0
        for site in sites:
            yrs = sorted({int(y) for y in sub.loc[sub["site"].eq(site), "year"].dropna()})
            y_centers.append((site, yrs, y))
            y_tick_pos.append(y)
            y_tick_lab.append(site)
            y += SITE_STEP
        layout[veg] = {
            "centers": y_centers,
            "ticks": y_tick_pos,
            "labels": y_tick_lab,
            "ymax": max(y - SITE_STEP, 0.0),
        }

    fig, axes = plt.subplots(
        len(COMP_METRICS),
        n_veg,
        figsize=(5.8 * n_veg, max(4.0, 0.48 * max_sites + 2.0) * len(COMP_METRICS)),
        sharex=False,
        squeeze=False,
    )

    for row_i, (col, phase, _) in enumerate(COMP_METRICS):
        panel_artists = []
        for col_i, veg in enumerate(vegs):
            ax = axes[row_i][col_i]
            info = layout[veg]
            sub = plot_df.loc[plot_df["veg"].eq(veg)].copy()
            sub["_year_i"] = pd.to_numeric(sub["year"], errors="coerce").astype("Int64")

            drawn = []
            for site, yrs, y0 in info["centers"]:
                site_rows = sub.loc[sub["site"].eq(site)]
                if 2023 in yrs and 2024 in yrs:
                    year_y = {2023: y0 - PAIR_HALF, 2024: y0 + PAIR_HALF}
                elif yrs:
                    year_y = {yrs[0]: y0}
                else:
                    year_y = {}

                for year, yy in year_y.items():
                    r = site_rows.loc[site_rows["_year_i"].eq(year)]
                    if r.empty or col not in r.columns:
                        continue
                    val = r.iloc[0][col]
                    if pd.isna(val):
                        continue
                    # ratio-1 so 0 = match GCC; keeps 0 at center like lag
                    delta = float(val) - 1.0
                    drawn.append((yy, delta, year))

            panel_artists.append((ax, info, drawn))

        for col_i, (ax, info, drawn) in enumerate(panel_artists):
            for yy, delta, year in drawn:
                color = YEAR_COLOR.get(year, "#555555")
                ax.hlines(yy, 0, delta, color=color, alpha=0.65, linewidth=1.6)
                ax.scatter(
                    [delta], [yy], color=color, s=40, zorder=3,
                    edgecolors="white", linewidths=0.4,
                )

            ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.8)
            ax.set_yticks(info["ticks"])
            ax.set_yticklabels(info["labels"], fontsize=7)
            ax.set_ylim(-0.55, info["ymax"] + 0.55)
            ax.invert_yaxis()
            ax.grid(True, axis="x", alpha=0.3)

            # per-panel: max(|neg|,|pos|) + 5; small panels stay tight, large can exceed 100
            lim = _sym_axis_lim([d for _, d, _ in drawn], pad=0.5, empty=0.5)
            _apply_centered_xlim(ax, lim)

            if row_i == 0:
                ax.set_title(f"{vegs[col_i]} (n_sites={len(info['labels'])})")
            ax.set_xlabel(phase)

        right_ax = axes[row_i][-1]
        right_ax.text(
            1.20, 0.95, phase,
            transform=right_ax.transAxes,
            ha="left", va="top", fontsize=11, fontweight="bold", clip_on=False,
        )

    fig.suptitle(
        f"{title_prefix} — Compression 2023 (blue) vs 2024 (green)",
        y=0.995,
    )
    fig.subplots_adjust(left=0.09, right=0.90, top=0.92, bottom=0.22, hspace=0.80, wspace=1.05)
    fig.legend(
        handles=[
            Line2D([0], [0], color=YEAR_COLOR[2023], lw=3, marker="o", label="2023"),
            Line2D([0], [0], color=YEAR_COLOR[2024], lw=3, marker="o", label="2024"),
            Line2D([0], [0], color="none", label="0 : same length as GCC"),
            Line2D([0], [0], color="none", label="+ : GVF longer (stretched) | − : GVF shorter (compressed)"),
            Line2D([0], [0], color="none", label=COMP_METRICS[0][2]),
            Line2D([0], [0], color="none", label=COMP_METRICS[1][2]),
        ],
        loc="upper center",
        bbox_to_anchor=(0.47, 0.18),
        ncol=1,
        frameon=True,
        fontsize=8,
    )
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight", pad_inches=0.45)
    plt.close(fig)
    if show:
        display(Image(filename=str(out_png)))
    print(f"Wrote {out_png}")


out_comp = ANOMALY_DIR / "lollipopPlot" / "Combined" / "Compression_2023_2024.png"
plot_combined_compression_years(_plot_df, out_comp, show=False)


combined compression pool n=53 | sites=39 | veg=['AG', 'DB', 'GR', 'SH']
Wrote /Applications/Home/All School/School Past/2026 Fall/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/lollipopPlot/Combined/Compression_2023_2024.png


## Findings notes

### GoldenSites 2023

Veg codes: DB = deciduous broadleaf, EN = evergreen needle, GR = grassland, AG = agriculture, SH = shrub, EB = evergreen broadleaf

1) Compression: 
    - For Agriculture lands (n=6), green-up compression value shows mostly value greater than 1 which indiate gvf being stretched relative to gcc and senescence compression value also shows mostly value greater than 1 which is also stretched.
    - For deciduous broadleaf (n=14), green-up compression value is all positive which mean stretched. for senescence compression value its half site being stretched and half site being compressed. (could also say 2-3 site is a almost exact match (no stretch or compress, value = 1))
    - For evergreen needle and Grassland (both n = 1) all of the green-up and senescence seem to say it's being stretched
2) Lag:
    - EN and GR lag is insignificant 
    - For Agriculture lands (n=6), the lag various between -50 day eariler vs 50 day later SOS
    - For deciduous broadleaf (n=14), most site is a few days eariler but a few site are 50 days later SOS
